<a href="https://colab.research.google.com/github/apk41910/gas_calculator/blob/main/gas_calculator_v4_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
R = 0.08206  # L·atm/(mol·K)

from scipy.optimize import brentq

def calc_P(V, n, T):
    P=n*R*T/V
    return P

def calc_V(P, n, T):
    V=n*R*T/P
    return V

def calc_n(P, V, T):
    n=P*V/R/T
    return n

def calc_T(P, V, n):
    T=P*V/n/R
    return T


def to_kelvin(value, unit):
    if unit == "k":
        return value
    elif unit == "c":
        return value + 273.15
    elif unit == "f":
        return (value - 32) * 5/9 + 273.15
    elif unit == "r":
        return value * 5/9


def to_base(var):
    number, unit = values[var]

    if var == "P":
        return number * to_atm[unit]
    elif var == "V":
        return number * to_L[unit]
    elif var == "n":
        return number * to_mol[unit]
    elif var == "T":
        return to_kelvin(number, unit)



to_atm = {
    "atm":  1,
    "pa":   1/101325,
    "kpa":  1/101.325,
    "mpa":  1/0.101325,
    "bar":  1/1.01325,
    "mmhg": 1/760,
    "torr": 1/760,
    "psi":  1/14.696,
  }

# 부피 → L
to_L = {
    "l":   1,
    "ml":  0.001,
    "cm3": 0.001,
    "m3":  1000,
    "ft3": 28.317,
    "gal": 3.7854,
}

# 몰수 → mol
to_mol = {
    "mol":  1,
    "mmol": 0.001,
    "kmol": 1000,
}


to_T = {
    "k": 1,
    "c": 1,
    "f": 1,
    "r": 1
}


# Tc, Pc: 임계온도(K), 임계압력(atm)
# omega: 이심인자(acentric factor)
# M: 분자량(g/mol)
# antoine: (A, B, C) — log10(P/mmHg) = A - B/(C + T), T는 °C
# antoine_range: antoine 계수가 유효한 온도 범위 (°C)

substances = {
    "co2":  {"name": "CO2",  "M": 44.01, "Tc": 304.2, "Pc": 72.9,  "omega": 0.225,
             "antoine": (7.5788, 865.71, 273.48),    "antoine_range": (-119, -69)},

    "n2":   {"name": "N2",   "M": 28.01, "Tc": 126.2, "Pc": 33.5,  "omega": 0.040,
             "antoine": (6.49457, 255.68, 266.55),   "antoine_range": (-210, -184)},

    "o2":   {"name": "O2",   "M": 32.00, "Tc": 154.6, "Pc": 49.8,  "omega": 0.022,
             "antoine": (6.69144, 319.013, 266.697), "antoine_range": (-219, -183)},

    "ch4":  {"name": "CH4",  "M": 16.04, "Tc": 190.6, "Pc": 45.4,  "omega": 0.011,
             "antoine": (6.61184, 389.93, 266.00),   "antoine_range": (-181, -152)},

    "h2o":  {"name": "H2O",  "M": 18.02, "Tc": 647.1, "Pc": 217.8, "omega": 0.345,
             "antoine": (8.07131, 1730.63, 233.426), "antoine_range": (1, 100)},

    "nh3":  {"name": "NH3",  "M": 17.03, "Tc": 405.5, "Pc": 111.3, "omega": 0.250,
             "antoine": (7.36050, 926.132, 240.17),  "antoine_range": (-83, 60)},

    "h2":   {"name": "H2",   "M": 2.016, "Tc": 33.2,  "Pc": 12.8,  "omega": -0.216,
             "antoine": (5.92088, 71.6153, 276.34),  "antoine_range": (-259, -253)},

    "c2h6": {"name": "C2H6", "M": 30.07, "Tc": 305.3, "Pc": 48.1,  "omega": 0.099,
             "antoine": (6.83452, 663.70, 256.47),   "antoine_range": (-143, -75)},
}



In [ ]:
class EOS:
    name = "EOS"

    def __init__(self, sub):
        self.sub = sub

    def ab(self):
        raise NotImplementedError

    def alpha(self, T):
        return 1.0

    def pressure(self, Vm, T):
        raise NotImplementedError

    def solve_Vm(self, P_target, T):
        return brentq(lambda Vm: self.pressure(Vm, T) - P_target, 0.05, 1000.0)

    def solve_T(self, P_target, Vm):
        return brentq(lambda T: self.pressure(Vm, T) - P_target, 1, 5000)


class Ideal(EOS):
    name = "Ideal"

    def pressure(self, Vm, T):
        return R*T/Vm

class VdW(EOS):
    name = "vdW"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 27*R**2*Tc**2/(64*Pc)
        b = R*Tc/(8*Pc)
        return a, b

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a/Vm**2
class RK(EOS):
    name = "RK"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        c = 2 ** (1/3)
        a = 1 / (9 * (c - 1)) * R**2 * Tc**2.5 / Pc
        b = (c - 1) / 3 * R * Tc / Pc        # ← 추가
        return a, b                           # ← 추가

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a/(T**0.5 * Vm * (Vm + b))



class SRK(EOS):
    name = "SRK"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 0.42748*R**2*Tc**2/Pc
        b = 0.08664*R*Tc/Pc
        return a, b

    def alpha(self, T):
        Tc = self.sub["Tc"]
        omega = self.sub["omega"]
        m = 0.480 + 1.574*omega - 0.176*omega**2
        return (1 + m*(1 - (T/Tc)**0.5))**2

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a*self.alpha(T)/(Vm*(Vm + b))

class PR(EOS):
    name = "PR"

    def ab(self):
        Tc, Pc = self.sub["Tc"], self.sub["Pc"]
        a = 0.45724*R**2*Tc**2/Pc
        b = 0.07780*R*Tc/Pc
        return a, b

    def alpha(self, T):
        Tc = self.sub["Tc"]
        omega = self.sub["omega"]
        kappa = 0.37464 + 1.54226*omega - 0.26992*omega**2
        return (1 + kappa*(1 - (T/Tc)**0.5))**2

    def pressure(self, Vm, T):
        a, b = self.ab()
        return R*T/(Vm - b) - a*self.alpha(T)/(Vm*(Vm + b) + b*(Vm - b))



import math

def vapor_pressure(T, sub):
    """T(K)에서의 증기압(atm)과 사용한 방법. 계산 불가면 (None, None)"""
    Tc, Pc, omega = sub["Tc"], sub["Pc"], sub["omega"]
    Tr = T / Tc

    # 조건 1: 임계온도 이상이면 증기압 자체가 없음
    if Tr >= 1:
        return None, None

    # 조건 2: Antoine 계수가 있고 범위 안이면 실측 기반 우선
    if "antoine" in sub:
        A, B, C = sub["antoine"]
        Tmin, Tmax = sub["antoine_range"]
        T_celsius = T - 273.15
        if Tmin <= T_celsius <= Tmax:
            return 10 ** (A - B/(C + T_celsius)) / 760, "Antoine"

    # 조건 3: 너무 저온이면 상관식도 못 씀
    if Tr < 0.3:
        return None, None

    # 조건 4: 그 외 → Ambrose-Walton
    tau = 1 - Tr
    f0 = (-5.97616*tau + 1.29874*tau**1.5 - 0.60394*tau**2.5 - 1.06841*tau**5) / Tr
    f1 = (-5.03365*tau + 1.11505*tau**1.5 - 5.41217*tau**2.5 - 7.46628*tau**5) / Tr
    f2 = (-0.64771*tau + 2.41539*tau**1.5 - 4.26979*tau**2.5 + 3.25259*tau**5) / Tr

    return Pc * math.exp(f0 + omega*f1 + omega**2 * f2), "Ambrose-Walton"


def check_phase(P, T, sub):
    """(상, 사용한 증기압 식) 을 return"""
    Tc, Pc = sub["Tc"], sub["Pc"]

    # 조건 1: 임계온도 위
    if T > Tc:
        return ("초임계" if P > Pc else "기체"), None

    # 조건 2: 증기압과 비교
    Psat, method = vapor_pressure(T, sub)

    if Psat is None:
        return "판별불가", None

    return ("기체" if P < Psat else "액체"), method

def print_result(label, value, unit, phase=None):
    """계산 결과 한 줄 출력. value가 None이면 '해 없음'"""
    if value is None:
        print(f"  {label:<6} {'해 없음':>12}   (기체 아님)")
        return

    line = f"  {label:<6} {value:>12.4f} {unit:<4}"
    if phase:
        line += f"   ({phase})"
    print(line)


def print_phase(P, T, sub):
    """계산 전 상 경고. (상, 사용한 증기압 식) return"""
    phase, method = check_phase(P, T, sub)
    if phase == "액체":
        print("  ※ 이 조건에서는 액체입니다. 아래 결과는 신뢰할 수 없습니다.")
    return phase, method

print(vapor_pressure(373.15, substances["h2o"]))   # 1 atm 근처 (물 끓는점)
print(vapor_pressure(298.15, substances["h2o"]))   # 0.031 근처
print(vapor_pressure(300, substances["co2"]))      # 67 근처
print(vapor_pressure(304.2, substances["co2"]))    # Tr=1이라 None

(1.000113643638067, 'Antoine')
(0.031166333622493707, 'Antoine')
(66.14249179055874, 'Ambrose-Walton')
(None, None)


In [ ]:
while True:
    name = input("물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): ").lower()

    if name == "q":
        break

    if name not in substances:
        print("등록되지 않은 물질입니다.")
        continue

    sub = substances[name]
    print(f"{sub['name']} 선택됨")

    print("사용 가능한 단위")
    print("  압력: atm, kPa, bar, mmHg, psi ...")
    print("  부피: L, mL, m3 ...")
    print("  몰수: mol, mmol, kmol")
    print("  온도: K, C, F")

    eos_list = [Ideal(sub), VdW(sub), RK(sub), SRK(sub), PR(sub)]

    while True:                                # 안쪽: 계산 반복
        text = input(f"[{sub['name']}] 값 3개 입력 (b: 물질 변경) > ")

        if text == "b":
            break                              # 안쪽만 종료 → 물질 선택으로

        values = {}
        has_error = False

        for item in text.split(","):
            try:
                number, unit = item.split()
                number = float(number)
                unit = unit.lower()

                if unit in to_atm:
                    values["P"] = (number, unit)
                elif unit in to_L:
                    values["V"] = (number, unit)
                elif unit in to_mol:
                    values["n"] = (number, unit)
                elif unit in to_T:
                    values["T"] = (number, unit)
                else:
                    print("모르는 단위입니다:", unit)
            except ValueError:
                print("입력 형식을 확인해주세요. 예: 200 kPa, 2 mol, 300 K")
                has_error = True
                break

        if has_error:
            continue

        missing = [v for v in ["P", "V", "n", "T"] if v not in values]

        if len(missing) != 1:
            print("값 3개를 입력해주세요.")
            continue

        target = missing[0]
        print("{}를 구하겠습니다".format(target))

        if target == "P":
            V, n, T = to_base("V"), to_base("n"), to_base("T")
            Vm = V / n

            for eos in eos_list:
                P = eos.pressure(Vm, T)
                phase, method = check_phase(P, T, sub)
                label = f"{phase}, {method}" if method else phase
                print_result(eos.name, P, "atm", label)

        elif target == "V":
            P, n, T = to_base("P"), to_base("n"), to_base("T")
            phase, method = print_phase(P, T, sub)
            label = f"{phase}, {method}" if method else phase

            for eos in eos_list:
                try:
                    print_result(eos.name, eos.solve_Vm(P, T) * n, "L", label)
                except ValueError:
                    print_result(eos.name, None, "L")

        elif target == "n":
            P, V, T = to_base("P"), to_base("V"), to_base("T")
            phase, method = print_phase(P, T, sub)
            label = f"{phase}, {method}" if method else phase

            for eos in eos_list:
                try:
                    print_result(eos.name, V / eos.solve_Vm(P, T), "mol", label)
                except ValueError:
                    print_result(eos.name, None, "mol")

        elif target == "T":
            P, V, n = to_base("P"), to_base("V"), to_base("n")
            Vm = V / n

            for eos in eos_list:
                try:
                    T = eos.solve_T(P, Vm)
                    phase, method = check_phase(P, T, sub)
                    label = f"{phase}, {method}" if method else phase
                    print_result(eos.name, T, "K", label)
                except ValueError:
                    print_result(eos.name, None, "K")

물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): h2o
H2O 선택됨
사용 가능한 단위
  압력: atm, kPa, bar, mmHg, psi ...
  부피: L, mL, m3 ...
  몰수: mol, mmol, kmol
  온도: K, C, F
값 3개 입력 (b: 뒤로가기) >  b
입력 형식을 확인해주세요. 예: 200 kPa, 2 mol, 300 K
값 3개 입력 (b: 뒤로가기) > co2
입력 형식을 확인해주세요. 예: 200 kPa, 2 mol, 300 K
값 3개 입력 (b: 뒤로가기) > 0.5 L, 1 mol, 300 K
P를 구하겠습니다
  Ideal       49.2360 atm    (액체, Antoine)
  vdW         30.5848 atm    (액체, Antoine)
  RK          20.2133 atm    (액체, Antoine)
  SRK         14.4118 atm    (액체, Antoine)
  PR          15.1215 atm    (액체, Antoine)
값 3개 입력 (b: 뒤로가기) > b
물질 선택 (co2, n2, o2, ch4, h2o, nh3, h2, c2h6 | q: 종료): h2o
H2O 선택됨
사용 가능한 단위
  압력: atm, kPa, bar, mmHg, psi ...
  부피: L, mL, m3 ...
  몰수: mol, mmol, kmol
  온도: K, C, F
값 3개 입력 (b: 뒤로가기) > 1 atm, 1 mol, 25 c
V를 구하겠습니다
  ※ 이 조건에서는 액체입니다. 아래 결과는 신뢰할 수 없습니다.
  Ideal       24.4662 L      (액체, Antoine)
  vdW            해 없음   (기체 아님)
  RK             해 없음   (기체 아님)
  SRK            해 없음   (기체 아님)
  PR             해 없음   (기